# 04 - Regressao linear: prever time to hireRegressao linear e usada quando queremos prever um numero. Neste notebook, o numero e `time_to_hire_dias`, ou seja, quantos dias uma vaga levou da abertura ate a admissao.Este exemplo e didatico. O dataset e pequeno, entao o foco e entender o processo, nao criar um modelo pronto para uso real.

In [ ]:
# pathlib localiza arquivos.from pathlib import Path# pandas trabalha com tabelas.import pandas as pd# ColumnTransformer permite tratar colunas numericas e categoricas de formas diferentes.from sklearn.compose import ColumnTransformer# LinearRegression e o modelo de regressao linear.from sklearn.linear_model import LinearRegression# Metricas para avaliar erro do modelo.from sklearn.metrics import mean_absolute_error, r2_score# train_test_split separa dados em treino e teste.from sklearn.model_selection import train_test_split# Pipeline junta preprocessamento e modelo em uma unica sequencia.from sklearn.pipeline import Pipeline# OneHotEncoder transforma categorias em colunas numericas.from sklearn.preprocessing import OneHotEncoder# Localizamos e abrimos a base.BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()vagas = pd.read_csv(BASE_DIR / "dados" / "vagas_recrutamento.csv")# Conferimos as primeiras linhas.vagas.head()

## 1. Definir alvo e variaveis explicativasO alvo e o que queremos prever. As variaveis explicativas sao as informacoes usadas para fazer a previsao.

In [ ]:
# Esta e a coluna que queremos prever.alvo = "time_to_hire_dias"# Variaveis numericas ja sao numeros e podem entrar no modelo diretamente.variaveis_numericas = ["candidatos", "entrevistas", "ofertas", "custo_divulgacao"]# Variaveis categoricas sao textos. Precisam ser convertidas antes do modelo.variaveis_categoricas = ["area", "senioridade", "source_of_hire", "recrutador"]# Juntamos todas as variaveis explicativas em uma lista.variaveis = variaveis_numericas + variaveis_categoricas# X guarda as variaveis explicativas.X = vagas[variaveis]# y guarda a variavel alvo.y = vagas[alvo]X.head()

## 2. Preparar o preprocessamentoModelos matematicos nao entendem texto como `LinkedIn` ou `RH`. O `OneHotEncoder` transforma cada categoria em colunas de 0 e 1.

In [ ]:
# ColumnTransformer aplica transformacoes diferentes para grupos de colunas.preprocessamento = ColumnTransformer(    transformers=[        # Nas categorias, aplicamos OneHotEncoder.        ("categorias", OneHotEncoder(handle_unknown="ignore"), variaveis_categoricas),        # Nos numeros, usamos passthrough, ou seja, passamos como estao.        ("numeros", "passthrough", variaveis_numericas),    ])# Pipeline executa primeiro o preprocessamento e depois a regressao.modelo = Pipeline(    steps=[        ("preprocessamento", preprocessamento),        ("regressao", LinearRegression()),    ])

## 3. Separar treino e testeTreino e usado para aprender. Teste e usado para avaliar com dados que o modelo nao viu durante o treino.

In [ ]:
# test_size=0.25 significa que 25% dos dados ficarao para teste.# random_state fixa a aleatoriedade para o resultado ser reproduzivel.X_treino, X_teste, y_treino, y_teste = train_test_split(    X,    y,    test_size=0.25,    random_state=42,)# fit treina o modelo com os dados de treino.modelo.fit(X_treino, y_treino)# predict gera previsoes para os dados de teste.previsoes = modelo.predict(X_teste)

## 4. Comparar real versus previstoA tabela abaixo mostra, para cada vaga do teste, o valor real, o valor previsto e o erro absoluto.

In [ ]:
# Criamos uma tabela para comparar resultados.resultado = pd.DataFrame({    "real": y_teste.values,    "previsto": previsoes.round(1),})# Erro absoluto e a distancia entre real e previsto, ignorando sinal.resultado["erro_absoluto"] = (resultado["real"] - resultado["previsto"]).abs().round(1)resultado

## 5. Avaliar o modeloErro medio absoluto e facil de explicar: em media, quantos dias o modelo erra. R2 mede quanto da variacao foi explicada, mas e menos intuitivo para publico iniciante.

In [ ]:
# mean_absolute_error calcula a media dos erros absolutos.mae = mean_absolute_error(y_teste, previsoes)# r2_score calcula o R2.r2 = r2_score(y_teste, previsoes)print(f"Erro medio absoluto: {mae:.1f} dias")print(f"R2: {r2:.2f}")

## 6. Simular uma nova vagaAgora criamos uma vaga ficticia e pedimos uma previsao. Isso ajuda a entender como o modelo seria usado, mas nao transforma o resultado em certeza.

In [ ]:
# Criamos um DataFrame com uma unica nova vaga.# As colunas precisam ter os mesmos nomes usados no treino.nova_vaga = pd.DataFrame([    {        "candidatos": 70,        "entrevistas": 12,        "ofertas": 2,        "custo_divulgacao": 800,        "area": "RH",        "senioridade": "Pleno",        "source_of_hire": "LinkedIn",        "recrutador": "Ana",    }])# O modelo devolve uma previsao em dias.previsao = modelo.predict(nova_vaga)[0]print(f"Previsao didatica de time to hire: {previsao:.0f} dias")

## 7. Exercicios1. Remova `recrutador` das variaveis e rode novamente.2. Altere a `nova_vaga` para uma vaga de TI senior.3. Compare o erro medio absoluto antes e depois.4. Escreva uma frase executiva explicando o erro do modelo.5. Escreva uma limitacao etica ou metodologica deste modelo.